1. create list with customs ideas (one word a custom)

In [ ]:
list_of_customs = ["clown","Astronaut","Pirate","Ninja","Vampire","Witch","Superhero","Princess","Commercial pilot","Rock musician","king", "Cowboy"]


2. upload a portrait photo from a directory and save it into images directory

In [ ]:
from pathlib import Path
import shutil

# example: "C:\Users\User\Desktop\WhatsApp Image 2026-03-01 at 12.24.35.jpeg"

# Strip both whitespace AND double quotes
src_path = Path(input("Path to portrait file: ").strip().strip('"'))
# destination directory for the images
dest_dir = Path("../images")
# copy the file into your images directory
dest_path = dest_dir / src_path.name
print(f"Copying {src_path} to {dest_path}...")
shutil.copy2(src_path, dest_path)


2.1 show the image to user

In [ ]:

from IPython.display import Image
Image(filename= dest_path, width=300)  

3. send portrait image to llm (gpt5) and ask for a description of the portrait, save the description into a variable

In [ ]:
from openai import OpenAI                                     
import os 
import base64                                                                        
from dotenv import load_dotenv                                                    
load_dotenv() 

openai_api_key = os.getenv("OPENAI_API_KEY")  
openai_client = OpenAI (api_key=openai_api_key) 
client = openai_client

def encode_image(image_path):
	with open(image_path, "rb") as image_file:
		return base64.b64encode(image_file.read()).decode("utf-8")

image_path = dest_path

base64_image = encode_image(image_path)


response = client.chat.completions.create(
	model="gpt-5-mini-2025-08-07",
	messages=[
		{
			"role": "user",
			"content": [
				{
					"type": "text",
					"text": "give me a description of the image",
				},
				{
					"type": "image_url",
					"image_url": {"url": f"data:image/jpeg;base64,{base64_image}"},
				},
			],
		}
	],
)

print(response.choices[0].message.content)

image_description = response.choices[0].message.content



4. print the customs list to the user, ask the user to choose a custum, save the custom name into a variable

In [ ]:
print(list_of_customs)
person_costumes = input("Enter the costumes that you choose")
costumes = person_costumes

print(f"Your costume choices are: {costumes}")

5. send the description and the custom name to llm (gpt5) and ask for a realistic image descrition of the portrait object wearing the custom.
save the image description into a variable

In [ ]:
from pydantic import BaseModel
from openai import OpenAI

client = OpenAI()

class ImagePrompt(BaseModel):
    prompt: str

response = client.beta.chat.completions.parse(
    model="gpt-5-mini-2025-08-07",
    messages=[
        {
            "role": "user",
            "content": (
                f"Give me a description of a realistic image of {image_description} "
                f"dressed as a {costumes} in the same position and environment as "
                f"the original image. Maintain identical subject placement, "
                f"camera angle, framing, and perspective."
            ),
        }
    ],
    response_format=ImagePrompt,
)

new_costume_description = response.choices[0].message.parsed.prompt

print(f"New costume description: {new_costume_description}")

6. ספריית רפליקט

In [ ]:
import replicate                                                    # Import the replicate library for AI model inference
import requests                                                     # Import requests library for making HTTP requests
import time                                                         # Import time library for sleep functionality
import os                                                           # Import os library for environment variable access
from IPython.display import Image, display                          # Import Image and display from IPython.display to show images in Jupyter Notebook

from dotenv import load_dotenv  
load_dotenv ()                                                     # Import load_dotenv to load environment variables from a .env file

api_token = os.getenv("REPLICATE_API_KEY")                          # Get the Replicate API token from environment variables
os.environ["REPLICATE_API_TOKEN"] = api_token                       # Set the Replicate API token in the environment variables

if not api_token:                                                   # Check if the Replicate API token is not found
    print("⚠️ Warning: REPLICATE_API_TOKEN not found.")
    print("   Please ensure a .env file exists in the project root directory")
    print("   with the line: REPLICATE_API_TOKEN=your_actual_token")
else:                                                               # If the token is found, print success message
    print("✅ Replicate API Token loaded successfully.")

6.1 send the portrait image as a refernce to flux2.dev with the image description we made before, save the image to directory customs.

In [ ]:
output = replicate.run(
    "black-forest-labs/flux-kontext-pro",
    input={
        "prompt": new_costume_description,
        "input_image": image_path ,
        "aspect_ratio": "match_input_image",
        "output_format": "jpg",
        "safety_tolerance": 2
    }
)

# To write the file to disk:
with open(f"../customs/{costumes}_1.jpg", "wb") as file:
    file.write(output.read())

file_path = f"../customs/{costumes}_1.jpg"

    

6.2 : ננו בננה

In [ ]:
with open(image_path, "rb") as img:

    output = replicate.run(
    "google/nano-banana-2",
    input={
        "prompt": new_costume_description,
        "resolution": "1K",
        "image_input": [img],
        "aspect_ratio": "1:1",
        "image_search": False,
        "google_search": False,
        "output_format": "jpg"
    }
)


# To write the file to disk:
with open(f"../customs/{costumes}_2.jpg","wb") as file:
    file.write(output.read())

file_path = f"../customs/{costumes}_2.jpg"    

7. print image to user

In [ ]:

from IPython.display import Image
Image(filename= file_path, width=300) 